In [1]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
from ingest import ingest_faq_data,build_index
from rag_helper import RAGBase
from sqlitesearch import TextSearchIndex

load_dotenv()

system_instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function.
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

The question has to be about the course or its logistics, offtopic questions
shouldn't be answered. If the search return nothing,it's likely an off-topic question.
if you can't answer the question using FAQ don't do it yourself only use the facts from the FAQ database

At the end, ask if there are other areas that the user wants to explore.
""".strip()

docs = ingest_faq_data()
index = TextSearchIndex(text_fields=["question", "section", "answer"],
    keyword_fields=["course"],db_path="faq.db")

gemini_client = genai.Client()

config = types.GenerateContentConfig(
    temperature =0.2,
    system_instruction=system_instruction
)

assistant = RAGBase(index = index,llm_client = gemini_client,config = config ,model = "gemini-3.6-flash")
question = "I just discovered the course. Can i join it?"
answer = assistant.rag(question)
print(answer)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Yes, you can still join. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.


In [19]:
def search(question,num_results = 5):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
            question,
            boost_dict=boost_dict,
            filter_dict=filter_dict,
            num_results=num_results
    )


search_declaration = types.FunctionDeclaration(
    name="search",
    description="Search the FAQ database for entries matching the given query.",
    parameters_json_schema={
        "type":"object",
        "properties" : {
            "question": {
                "type":"string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required" : ["question"],
        "additionalProperties": False
    }
)

search_tool = types.Tool(function_declarations=[search_declaration])

In [21]:
tool_config = types.GenerateContentConfig(
    temperature =0.2,
    system_instruction=system_instruction,
    tools = [search_tool]
)

contents = [
    types.Content(role = "user", parts = [types.Part(text=question)])
]

response = gemini_client.models.generate_content(
    model = "gemini-3.6-flash",
    contents=contents,
    config = tool_config
)

response.function_calls

[FunctionCall(
   args={
     'question': 'Can I join the course after it started?'
   },
   id='call_38333',
   name='search'
 )]

In [26]:
call = response.function_calls[0]
args = call.args
print(args)
name = call.name
print(name)
print(response.candidates[0].content)

result = search(**args)
print(result)


contents.append(response.candidates[0].content)

contents.append(
    types.Content(
    role = "user",
    parts= [types.Part.from_function_response(
        name=call.name,
        response={"result": result},
    )
    ],
)
)

{'question': 'Can I join the course after it started?'}
search
parts=[Part(
  function_call=FunctionCall(
    args={
      'question': 'Can I join the course after it started?'
    },
    id='call_38333',
    name='search'
  ),
  thought_signature=b"\x12\xe2\x01\n\xdf\x01\x01i\x14}\x13\xd6\x96\xdf\xae\xdf\x95\x07\x83-z\xc9Q-\xb3n\xf3\x0b\xed\xd2\xdd\x8f\x15\xec\xcc\xc0%\xe7\xee<\x98N\xb1\x91\xba\xf6\x80\x87\xf8\xb3&rI\xce\xf3\xd0S\xbb\xcd\xa8'\xac\xc2Pq\x93a\xfd\xb2?\xd9\xd6\xbb\xa1u\x80E\x92\x82r>\xe8B\x9d8`Q\xd3\x1f\x90]\xef\xe7\xc7\xd3\x1e\xc8...'
)] role='model'
[{'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course. Can I still join?', 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}, {'id': '74eb249bbf', 'course': 'llm-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'I just discovered the course

In [27]:
response = gemini_client.models.generate_content(
    model = "gemini-3.6-flash",
    contents=contents,
    config = tool_config
)
print(response.text)

Yes, you can still join! However, if you want to receive a certificate, you will need to submit your project while project submissions are still being accepted.


Agentic Loop

In [30]:
def make_call(call):
    args = call.args

    if call.name == "search":
        result = search(**args)
    else:
        result = {"error": f"Unknown function called {call.name}"}

    return types.Part.from_function_response(name= call.name,response = {"result":result})


In [31]:
contents = [
    types.Content(role = "user", parts = [types.Part(text=question)])
]

response = gemini_client.models.generate_content(
    model = "gemini-3.6-flash",
    contents=contents,
    config = tool_config
)

answer = response.candidates[0].content

contents.append(answer)

function_response_parts = []

for part in answer.parts or []:
    if part.function_call:
        print("function_call:", part.function_call.name,part.function_call.args)
        function_response_parts.append(make_call(part.function_call))

    elif part.text:
        print("Assistant:")
        print(part.text)

if function_response_parts:
    contents.append(types.Content(role="user",parts = function_response_parts))

has_function_calls = len(function_response_parts) > 0

function_call: search {'question': 'Can I join the course after it started?'}


In [32]:
MODEL = "gemini-3.5-flash"
def agent_loop(instructions,question,model = MODEL,max_iterations=10):
    contents = [
        types.Content(role="user", parts = [types.Part(text=question)])
    ]

    config = types.GenerateContentConfig(
        system_instruction=instructions,
        tools=[search_tool]
    )

    last_answer = ""

    for it in range(1,max_iterations+1):
        print(f"iteration #{it}..")

        response = gemini_client.models.generate_content(
            model = model,
            contents = contents,
            config = config
        )

        model_content = response.candidates[0].content

        contents.append(model_content)

        function_response_parts = []

        for part in answer.parts or []:
            if part.function_call:
                print("function_call:", part.function_call.name,part.function_call.args)
                function_response_parts.append(make_call(part.function_call))

            elif part.text:
                print("Assistant:")
                print(part.text)
                last_answer = part.text

        if not function_response_parts:
            break

        contents.append(types.Content(role="user",parts=function_response_parts))

    return last_answer

In [33]:
agent_loop(system_instruction, "How do i run Olama locally?")

iteration #1..
function_call: search {'question': 'Can I join the course after it started?'}
iteration #2..


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 29.432068184s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'},

In [34]:
agent_loop(system_instruction,question)

iteration #1..


ClientError: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\nPlease retry in 26.485265101s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-3.5-flash'},